# 🐍 AI Travel Agent with Metacognition - Microsoft Agent Framework (Python)

## 📋 Scenario Overview

This notebook demonstrates **metacognition** in AI agents - the ability to remember and apply user preferences across conversations. Using the Microsoft Agent Framework, we'll build a travel booking agent that learns your flight time preferences and automatically applies them to future bookings.

**Key Concepts:**
- 🧠 **Metacognition**: Agent remembers preferences from past interactions
- 💾 **Conversation Memory**: Preferences persist across multiple requests
- 🎯 **Personalization**: Agent tailors recommendations based on learned preferences
- 🔄 **Context Awareness**: Agent references past decisions when making suggestions

## 🏗️ Technical Implementation

### Core Components
- **Agent Framework**: Python implementation of Microsoft's agent orchestration
- **Azure OpenAI**: GPT-4o-mini for natural language understanding
- **Microsoft Entra ID**: Secure keyless authentication
- **Tool Functions**: `get_destinations()` and `get_flight_times()`

### Metacognition Flow
```
First Booking:
User: "Book Barcelona flight" → Agent asks preference → User: "later flight"
                                        ↓
                            Agent stores: customer_preferences

Subsequent Bookings:
User: "Book Paris flight" → Agent recalls preference → "Based on your previous preference for later flights..."
```

## ⚙️ Prerequisites

**Required:**
```bash
pip install agent-framework-core azure-identity python-dotenv
```

**Environment Variables (.env):**
```env
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=gpt-4o-mini
AZURE_OPENAI_API_VERSION=2024-10-21
```

**Azure RBAC:** "Cognitive Services OpenAI User" role required

Let's build an intelligent agent with memory! 🌟

In [ ]:
# Package check - Install manually if needed: pip install agent-framework-core -U
try:
    import agent_framework
    print("✓ agent-framework-core is installed")
except ImportError:
    print("❌ Please install: pip install agent-framework-core -U")
    raise

In [ ]:
# 📦 Import Required Libraries
import os
from dotenv import load_dotenv
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import InteractiveBrowserCredential

In [ ]:
# 🔧 Load Environment Variables
load_dotenv(override=True)

# Verify configuration
print(f"✓ AZURE_OPENAI_ENDPOINT: {os.environ.get('AZURE_OPENAI_ENDPOINT')}")
print(f"✓ AZURE_OPENAI_CHAT_DEPLOYMENT_NAME: {os.environ.get('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")
print(f"✓ AZURE_OPENAI_API_VERSION: {os.environ.get('AZURE_OPENAI_API_VERSION')}")
print(f"AZURE_OPENAI_API_KEY: {'⚠️  SET (removing for Entra ID auth)' if os.environ.get('AZURE_OPENAI_API_KEY') else '✓ Not set (using Entra ID)'}")

# Critical: Remove API key if present to force Entra ID authentication
if os.environ.get('AZURE_OPENAI_API_KEY'):
    del os.environ['AZURE_OPENAI_API_KEY']
    print("✓ API key removed - using Entra ID authentication")

In [ ]:
# 🎲 Tool Functions: Destination and Flight Information
# These functions will be available to the agent as tools

def get_destinations() -> str:
    """Provides a list of vacation destinations.
    
    Returns:
        str: List of available vacation destinations
    """
    return """
    Barcelona, Spain
    Paris, France
    Berlin, Germany
    Tokyo, Japan
    New York, USA
    """

def get_flight_times(destination: str) -> str:
    """Provides available flight times for a destination.
    
    Args:
        destination: The destination to check flight times for
    
    Returns:
        str: Flight times for the specified destination
    """
    flight_times = {
        "Barcelona": ["08:30 AM", "02:15 PM", "10:45 PM"],
        "Paris": ["06:45 AM", "12:30 PM", "07:15 PM"],
        "Berlin": ["07:20 AM", "01:45 PM", "09:30 PM"],
        "Tokyo": ["11:00 AM", "05:30 PM", "11:55 PM"],
        "New York": ["05:15 AM", "03:00 PM", "08:45 PM"]
    }
    
    # Extract just the city name from input that might contain country
    city = destination.split(',')[0].strip()
    
    if city in flight_times:
        times = ", ".join(flight_times[city])
        return f"Flight times for {city}: {times}"
    else:
        return f"No flight information available for {city}."

print("✓ Tool functions defined: get_destinations(), get_flight_times()")

In [ ]:
# 🔗 Create Azure OpenAI Chat Client with Browser Authentication
# Uses Microsoft Entra ID for secure, keyless authentication

print("🔐 Authenticating with Azure...")
print("   A browser window will open for you to sign in")

# Create credential that will open browser for authentication
credential = InteractiveBrowserCredential()

# Create Azure OpenAI client
openai_chat_client = AzureOpenAIChatClient(
    credential=credential,
    endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION")
)

print(f"✓ Connected to Azure OpenAI")
print(f"  Endpoint: {os.environ.get('AZURE_OPENAI_ENDPOINT')}")
print(f"  Deployment: {os.environ.get('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")

In [ ]:
# 🤖 Create the Travel Agent with Metacognition Instructions
# The agent instructions explicitly tell it to remember and apply user preferences

AGENT_INSTRUCTIONS = """
You are Flight Booking Agent that provides information about available flights and gives travel activity suggestions when asked.
Travel activity suggestions should be specific to customer, location and amount of time at location.

You have access to the following tools to help users plan their trips:
1. get_destinations: Returns a list of available vacation destinations that users can choose from.
2. get_flight_times: Provides available flight times for specific destinations.

METACOGNITION - Your process for remembering and applying user preferences:
- When users first inquire about flight booking with no prior history, ask for their preferred flight time ONCE.
- MAINTAIN a customer_preferences object throughout the conversation to track preferred flight times.
- When a user books a flight to any destination, RECORD their chosen flight time in the customer_preferences object.
- For ALL subsequent flight inquiries to ANY destination, AUTOMATICALLY apply their existing preferred flight time without asking.
- NEVER ask about time preferences again after they've been established for any destination.
- When suggesting flights for a new destination, explicitly say: "Based on your previous preference for [time] flights, I recommend..."
- Only after showing options matching their preferred time, ask if they want to see alternative times.
- After each booking, UPDATE the customer_preferences object with any new information.
- ALWAYS mention which specific preference you used when making a suggestion.

Guidelines:
- Use the exact destination names when using tools (Barcelona, Paris, Berlin, Tokyo, New York)
- Respond in a helpful and enthusiastic manner about travel possibilities
- Always seek feedback to ensure your suggestions meet the user's expectations
- Acknowledge when a request falls outside your capabilities
- For better formatting, always display flight times in a list format
- When giving any timed suggestions, reflect if the time frames are reasonable. Respond again if not.

Your goal is to help users explore vacation options efficiently and make informed travel decisions by understanding their preferences and providing tailored recommendations.
"""

agent = ChatAgent(
    chat_client=openai_chat_client,
    instructions=AGENT_INSTRUCTIONS,
    tools=[get_destinations, get_flight_times]
)

print("✓ Travel agent created with metacognition capabilities")

In [ ]:
# 💬 First Conversation: Establish User Preferences
# Watch how the agent learns and stores the user's preference for "later flights"

from IPython.display import display, HTML
import asyncio

user_inputs = [
    "Book me a flight to Barcelona",
    "I prefer a later flight",
    "That is too late, choose the earliest flight",
    "I want to leave the same day, give me some suggestions of things to do in Barcelona during my layover if I take the last flight out",
    "I am stressed this wont be enough time"
]

print("🎯 Starting first conversation: Learning user preferences...\n")

async def run_conversation():
    for user_input in user_inputs:
        # Display user message
        html_output = (
            f"<div style='margin-bottom:10px'>" 
            f"<div style='font-weight:bold; color:#0066cc;'>👤 User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )
        
        # Get agent response (await the async call)
        response = await agent.run(user_input)
        
        # Display agent response
        html_output += (
            f"<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold; color:#009900;'>🤖 TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{response}</div></div><hr>"
        )
        
        display(HTML(html_output))

# Run the async conversation
await run_conversation()

print("\n✓ First conversation complete. Agent has learned user preferences.")

In [ ]:
# 🧠 Second Conversation: Demonstrate Metacognition
# The agent should remember the preference for later flights and apply it automatically!

user_input = "Book me a flight to Paris"

print("🧠 Testing metacognition: New destination, same conversation...\n")
print("💡 Watch for the agent to say: 'Based on your previous preference for later flights...'\n")

# Display user message
html_output = (
    f"<div style='margin-bottom:10px'>" 
    f"<div style='font-weight:bold; color:#0066cc;'>👤 User:</div>"
    f"<div style='margin-left:20px'>{user_input}</div></div>"
)

# Get agent response (await the async call)
response = await agent.run(user_input)

# Display agent response with highlighting
html_output += (
    f"<div style='margin-bottom:20px; background-color:#ffffcc; padding:10px; border-radius:5px;'>"
    f"<div style='font-weight:bold; color:#009900;'>🤖 TravelAgent (with memory):</div>"
    f"<div style='margin-left:20px; white-space:pre-wrap'>{response}</div></div>"
)

display(HTML(html_output))

print("\n✅ Metacognition demonstrated! The agent remembered preferences from the Barcelona booking.")

## 🎓 Key Takeaways: Metacognition in AI Agents

**What We Demonstrated:**
1. ✅ **Preference Learning**: Agent asked for preference once, then stored it
2. ✅ **Cross-Context Memory**: Preference from Barcelona applied to Paris booking
3. ✅ **Explicit Recall**: Agent stated "Based on your previous preference..."
4. ✅ **Personalization**: Recommendations tailored to learned preferences

**How It Works:**
- Agent maintains conversation history in memory
- System instructions tell agent to track `customer_preferences`
- LLM's context window preserves preference across multiple turns
- Agent references past decisions when making new recommendations

**Real-World Applications:**
- 🛒 **E-commerce**: Remember size, color, brand preferences
- 🏥 **Healthcare**: Recall patient allergies, medication preferences
- 💼 **Customer Service**: Remember communication preferences, history
- 🎯 **Personalization**: Build user profiles over time

**Limitations:**
- Memory is session-based (lost when conversation ends)
- For persistent memory, see **Lesson 13: Agent Memory**
- Context window limits long-term preference storage

**Next Steps:**
- Explore Lesson 13 for persistent memory with vector databases
- Implement preference storage in external systems (Cosmos DB, etc.)
- Add preference update mechanisms ("forget this preference")

---

**🎉 Congratulations!** You've built an AI agent with metacognitive capabilities using Microsoft Agent Framework!